<a href="https://colab.research.google.com/github/sreenadhyadav883/7006SCN2526MAYJUL-7006SCN_SYA_16073319/blob/main/notebooks/Task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# Run this cell once at the start of a fresh Colab runtime
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq > /dev/null
!pip install -q -U "pyspark[connect]~=4.0.0" findspark


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [12]:
import os
import findspark

# Java path used by Colab after installing OpenJDK 17
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
findspark.init()
from pyspark.sql import SparkSession

# Stop an old Spark session if the cell is re-run
try:
    spark.stop()
except NameError:
    pass
spark = (
    SparkSession.builder
    .appName("7006SCN_PySpark")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark

In [4]:
# Loading the dataset from the public mirror
# Used -q (quiet) so it doesn't spam the googlecolab output with download logs
!wget -q https://datasets-documentation.s3.eu-west-3.amazonaws.com/amazon_reviews/amazon_reviews_2015.snappy.parquet

In [13]:
import time
from pyspark.sql.functions import col, when
from pyspark.sql.types import StringType
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml import Pipeline

print("Loading and preparing data...")
#Re-load and Clean Data
df = spark.read.parquet("amazon_reviews_2015.snappy.parquet")
clean_df = df.dropna(subset=["review_body", "star_rating"])
clean_df = clean_df.withColumn("review_body", col("review_body").cast("string"))
final_df = clean_df.withColumn("label", when(col("star_rating") >= 4, 1).otherwise(0))

#Down-sample to 1% (approx 420,000 rows) so that google colab 12gb memory dont crash and throw a "Out of Memory" Error
sampled_df = final_df.sample(withReplacement=False, fraction=0.01, seed=42)
print(f"Sampled Dataset Size for Model Training: {sampled_df.count():,} rows")

#Reducing max features from 10,000 to 5,000
tokenizer = Tokenizer(inputCol="review_body", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
hashing_tf = HashingTF(inputCol="filtered_words", outputCol="rawFeatures", numFeatures=5000)
idf = IDF(inputCol="rawFeatures", outputCol="features")
pipeline = Pipeline(stages=[tokenizer, remover, hashing_tf, idf])
pipeline_model = pipeline.fit(sampled_df)
ml_data = pipeline_model.transform(sampled_df)

#Train/Test Split of 80:20 ratio
train_data, test_data = ml_data.randomSplit([0.8, 0.2], seed=42)
#Cache the training data in memory to dramatically speed up Cross-Validation
train_data.cache()

print("SETUP COMPLETE")
print(f"Training set: {train_data.count():,} rows")
print(f"Testing set: {test_data.count():,} rows")

Loading and preparing data...
Sampled Dataset Size for Model Training: 419,167 rows
SETUP COMPLETE
Training set: 335,427 rows
Testing set: 83,740 rows


In [14]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import time

print("Model 1:Logistic Regression")

#Initializing the Model
lr = LogisticRegression(featuresCol="features", labelCol="label")

#Building the Hyperparameter Grid
#Testing two different regularization parameters (0.01 and 0.1)
paramGrid_lr = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1]) \
    .build()
#Define "the Evaluator"
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

#Setingup the CrossValidator (3-fold cross validation)
#The operation here will train the model 6 separate times (3 folds*2 parameters)
cv_lr = CrossValidator(estimator=lr,estimatorParamMaps=paramGrid_lr,evaluator=evaluator,numFolds=3,seed=42)

#Model being trained and Time is being tracked!
print("Training started....")
start_time = time.time()
#Fitting the CrossValidator to find the best model
cv_lr_model = cv_lr.fit(train_data)
end_time = time.time()
lr_training_time = end_time - start_time

print("MODEL 1 Logistic Regression Completed")
print(f"Logistic Regression Training Time: {lr_training_time:.2f} seconds")
print(f"Best Regularization Parameter (regParam): {cv_lr_model.bestModel._java_obj.getRegParam()}")

Model 1:Logistic Regression
Training started....
MODEL 1 Logistic Regression Completed
Logistic Regression Training Time: 430.07 seconds
Best Regularization Parameter (regParam): 0.1
